# Learning pandas

Organized from my original `pandas_fundamentals.ipynb`, based on the [Kaggle pandas course](https://www.kaggle.com/learn/pandas). This is course practice, not an independent portfolio project. Original solution approaches and notes about using references are retained; indexing, explanatory, and portability fixes have been applied.

## Run this notebook

Requires Python, pandas, NumPy, and a Jupyter-compatible editor. Run cells from top to bottom. The default uses deterministic **synthetic sample data**, so no download or account is needed and results will differ from Kaggle. CSV export examples write to `outputs/` beside the notebook.

For the original data, set `USE_SAMPLE_DATA = False` and point `DATA_DIR` to your CSV folder. Expected filenames are `california_housing_test.csv`, `california_housing_train.csv`, `winemag-data-130k-v2.csv`, `CAvideos.csv`, `GBvideos.csv`, `gaming.csv`, and `movies.csv`; each notebook only loads the files its sections need. For the Reddit exercise, copy the two subreddit CSVs into this folder. No automatic dataset download is performed.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Default: reproducible SYNTHETIC examples, not the original Kaggle datasets.
# To use your CSVs, set USE_SAMPLE_DATA = False and update DATA_DIR.
USE_SAMPLE_DATA = True
DATA_DIR = Path("sample_data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 12)

def load_csv(filename, **kwargs):
    if not USE_SAMPLE_DATA:
        path = DATA_DIR / filename
        if not path.exists():
            raise FileNotFoundError(f"Place {filename} in {DATA_DIR.resolve()} or enable sample data.")
        return pd.read_csv(path, **kwargs)
    rng = np.random.default_rng(42)
    rows = 150
    if filename.startswith("california_housing"):
        return pd.DataFrame({
            "longitude": rng.uniform(-124, -114, rows),
            "latitude": rng.uniform(32, 42, rows),
            "housing_median_age": rng.integers(1, 53, rows),
            "total_rooms": rng.integers(100, 5000, rows),
            "total_bedrooms": rng.integers(30, 1000, rows),
            "population": rng.integers(50, 5000, rows),
            "households": rng.integers(20, 900, rows),
            "median_income": rng.uniform(1, 10, rows),
            "median_house_value": rng.integers(50000, 500000, rows),
        })
    if filename == "winemag-data-130k-v2.csv":
        # Synthetic tabular review records for practicing pandas operations.
        frame = pd.DataFrame({
            "country": np.resize(["Italy", "Canada", "Australia", "New Zealand", "US"], rows),
            "province": np.resize(["North", "South", "East"], rows),
            "region_1": np.resize(["Region A", "Region B", None], rows),
            "region_2": np.resize([None, "Area 1", "Area 2"], rows),
            "points": rng.integers(80, 101, rows),
            "price": rng.integers(10, 80, rows).astype(float),
            "description": np.resize(["tropical example", "fruity example", "plain example"], rows),
            "taster_name": np.resize(["Reviewer A", "Reviewer B", "Reviewer C"], rows),
            "taster_twitter_handle": np.resize(["@kerinokeefe", "@reviewer_b", "@reviewer_c"], rows),
            "title": [f"Sample record {i}" for i in range(rows)],
            "winery": np.resize(["Source A", "Source B", "Source C"], rows),
            "variety": np.resize(["Type A", "Type B", "Type C"], rows),
        })
        frame.loc[::13, "price"] = np.nan
        frame.loc[::17, "country"] = None
        return frame
    if filename in {"CAvideos.csv", "GBvideos.csv"}:
        return pd.DataFrame({
            "title": ["Shared video", "Canada video" if filename == "CAvideos.csv" else "UK video"],
            "trending_date": ["26.01.09", "26.01.09"], "views": [1000, 2000],
        })
    if filename in {"gaming.csv", "movies.csv"}:
        return pd.DataFrame({"product": ["Example A", "Example B"], "mentions": [3, 7]})
    raise ValueError(f"No synthetic sample defined for {filename}")

print("Data mode:", "SYNTHETIC SAMPLE — results differ from Kaggle" if USE_SAMPLE_DATA else "LOCAL CSV FILES")
reviews = load_csv("winemag-data-130k-v2.csv", index_col=0)


## Pandas Introduction

Pandas is a Python library used for data analysis as well as used for preparing data for machine learning. There are two main objects in pandas:
- DataFrame
- Series

## DataFrame

A DataFrame is a table which contains individual entries. The value of each entry may be datatypes such as integers or strings. Each of which corresponds to a row (or record) and a column.

In [ ]:
# simple DataFrame example
import pandas as pd
# Key's are the column names, the values are a list of entries for each
pd.DataFrame({'Yes': [50, 21], 'No': [131, 2]})


In [ ]:
# String df
pd.DataFrame({'Bob': ['I liked it.', 'It was awful.'], 'Sue': ['Pretty good.', 'Bland.']})

In [ ]:
# Notice that the df's above automatically assign the row labels to an ascending
# count (0, 1, 2, 3,...) however we can also assign our own values using index
# parameter in the constructor

pd.DataFrame({'Bob': ['I liked it.', 'It was awful.'],
              'Sue': ['Pretty good.', 'Bland.']},
             index=['Product A', 'Product B'])

## Series

A series is a sequence of data values. A series is essentially a single column of a data frame

In [ ]:
# Can create a series using a list
pd.Series([1, 2, 3, 4, 5])

In [ ]:
# Can assign row labels using an index parameter
# Can assign the column name using name
# Note the lack of curly braces unlike DataFrames
pd.Series([30, 35, 40], index=['2015 Sales', '2016 Sales', '2017 Sales'], name='Product A')

## Reading data files (CSV Files)

A csv (Comma-Separated Values) file is a table of values seperated by commas, hence the name.

In [ ]:
# Can use the load_csv() function to read data into a DataFrame
# Note the pd.read_csv has 30 optional parameters to specify
california_housing = load_csv("california_housing_test.csv")

# Can use shape to get the dimensions of the resulting DataFrame
california_housing.shape

In [ ]:

# Can also get a preview of the data using head() which gets the first five rows
california_housing.head()

## Exporting Pandas DataFrame to a CSV file

Pandas provides to_csv() function to export a DataFrame into a CSV file.

In [ ]:
scores = {'Name': ['a', 'b', 'c', 'd'],
          'Score': [90, 80, 95, 20]}

df = pd.DataFrame(scores)

df.to_csv(OUTPUT_DIR / "sample_scores.csv", index=False)

## Indexing, Selecting & Assigning

This section discusses how to select specific values from DataFrames and Series

In [ ]:
california_housing_train = load_csv('california_housing_train.csv')
# california_housing_train.head()

# Can access the property of an object by accessing it as an attribute
# Eg california_housing_train has the total_rooms property, which we can access below
california_housing_train.total_rooms

In [ ]:
# Python dictionary values can be accesseed using the indexing [] operator
# We can do the same with columns in a DataFrame

california_housing_train['population']

In [ ]:
# Can also use another [] for accessing specific series from a DataFrame
california_housing_train['households'][0]

## Indexing in pandas

Use `df.loc[row_labels, column_labels]` for labels and `df.iloc[row_positions, column_positions]` for integer positions. Both use row-first, column-second arguments. This contrasts with chained column-first selection such as `df["column"][label]`, not with Python indexing in general.

In [ ]:

# This selects first row of data from the DataFrame
california_housing_train.iloc[0]

In [ ]:
# To get a column with iloc, we do the following
california_housing_train.iloc[:, 0] # All rows, and column 0

In [ ]:
# On its own, : means "everything". Combined with other values it indicates a range
# To select the longitude from just the first, second, and third row, we would do:
california_housing_train.iloc[:3, 0]

In [ ]:
# Or just to get the second and third entries:
california_housing_train.iloc[1:3, 0] # 1 = start at second row, 3 = stop before fourth row

In [ ]:
# Can also pass a list into iloc too!
california_housing_train.iloc[[0, 1, 2], 0]

In [ ]:
# Can also use negative numbers to count forwards from the end of the values
# Here we use it to get the last five elements of the dataset
california_housing_train.iloc[-5:]

## Label-based selection

Besides iloc there is the loc operator: label-based selection. It uses the data index value rather than the position for accessing values

In [ ]:
# This selects the first entry in the DataFrame
# [row index, 'column_value_name']
california_housing_train.loc[0, 'longitude']

In [ ]:
# Can also pass lists as parameters
california_housing_train.loc[:, ['total_rooms', 'total_bedrooms', 'households']]

## Contrasting loc and iloc

Notice how iloc uses Python stdlib indexing scheme, where the first element of the range is included and last is excluded.

E.g. iloc 0:10 will select entries 0,...,9. While loc in contrast indexes inclusively, So with loc 0:10 will select entries 0,...,10

## Manipulating the index

`set_index()` returns a DataFrame using the chosen column as its index. Assign the result to retain it. Index objects themselves are immutable; a DataFrame can be given a replacement index.

In [ ]:
california_housing_train.set_index("housing_median_age")

## Conditional Selection

Can ask questions about the data through conditions.

In [ ]:
# For example can select the houses whose median house value is above 80,000
california_housing_train.loc[california_housing_train.median_house_value >= 80000]

In [ ]:
# Can also use & to bring multiple conditions together at once
# & ensures to only show rows which satisfy both conditions only
# | can be used to show rows which satisfy one or the other condition, or satisfy both
california_housing_train.loc[(california_housing_train.median_house_value >= 80000)
                             & (california_housing_train.latitude >= 40)]

In [ ]:
# Pandas comes with a few built-in conditional selectors
# First is isin which lets you select whose value "is in" a list of values

# Selects rows whose housing median age is 33, 17, or 19
california_housing_train.loc[california_housing_train.housing_median_age.isin([33, 17, 19])]

In [ ]:
# Another built-in conditional selector is isnull and notnull
california_housing_train.loc[california_housing_train.latitude.notnull()]

In [ ]:
# Eg of isnull
california_housing_train.loc[california_housing_train.latitude.isnull()]

## Assigning Data

Can also assign data to a dataframe, which is akin to assigning data to a dictionary

In [ ]:
# Can assign data using a constant value
california_housing_train['buyer'] = 'everyone'
california_housing_train['buyer']

In [ ]:
# Can also assign data using iterable values
california_housing_train['index_backwards'] = range(len(california_housing_train), 0, -1)
california_housing_train['index_backwards']

# Summary Functions and Maps

There are cases where we need to reformat the data selected from DataFrames or Series. This section covers different operations that can be applied to data to get the input "just right".

Summary functions restructure the data in some useful way, consider the describe() method:

In [ ]:
# Generates high level summary of the attributes of a given column
reviews.points.describe()

In [ ]:
# Can generate a summary for string data as well
reviews.taster_name.describe()

In [ ]:
# Can also get specific attributes of column data as well
print(f'Mean of points: {reviews.points.mean()}')
print(f'List of unique taster names: {reviews.taster_name.unique()}')

In [ ]:
# To see a list of unique values and how often they occur in the dataset
reviews.taster_name.value_counts()

## Maps

A map is a function that takes one set of values and "maps" them to another set of values.

Maps are used for tranforming data from its current format to another format we want later.

map() is one of the two main mapping methods

In [ ]:
# Can remean the scores the wine received to 0
reviews_points_mean = reviews.points.mean()
reviews.points.map(lambda p: p - reviews_points_mean)

# Note: The function passed to map() should expect a single value from the
# series (a point value) and return a transformed version of that value.

# Map() returns a new Series where all the values have been transformed
# by your function

`DataFrame.apply(..., axis="columns")` passes each row to a function. With the default axis it processes columns.

In [ ]:
def remean_points(row):
    return row["points"] - reviews_points_mean

reviews.apply(remean_points, axis="columns")


`map()` returns a transformed Series. `apply()` can return a Series or DataFrame depending on the function. Prefer functions that return values without mutating the objects passed to them.

Pandas aligns Series by index labels during arithmetic and string combination. Equal length alone does not guarantee corresponding entries will match.

In [ ]:
reviews.country + " - " + reviews.region_1

# Note that while built in functions like above are faster than map()
# or apply(), they are not capable of doing the advanced things that they
# can do, such as applying conditional logic, which cant be done with
# addition and subtraction alone

## Grouping and Sorting

Maps allowed us to transform data in DataFrame's and Series one value at a time for an entire column.

However grouping allows us to collect data into a single unit then allows us to do something specific to the entire group.

Each group can be thought of as being a slice of a DataFrame containing only data with values that match

### Groupwise analysis

Can replicate value_counts() function using count() on a group

In [ ]:
reviews.groupby('points').points.count()

In [ ]:
# Can also use other summary functions
# For example, to get the cheapest wine in each point value category:
reviews.groupby('points').price.min()

In [ ]:
# First title per source group.
reviews.groupby('winery')['title'].first()


In [ ]:
# Highest-scoring row in each country/province group.
best_indices = reviews.groupby(['country', 'province'])['points'].idxmax()
reviews.loc[best_indices]


In [ ]:
# agg() lets you run multiple functions on a DataFrame simultaneously
reviews.groupby(['country']).price.agg(['size', 'min', 'max'])

### Multi-indexes

groupby() allows us to group by multiple indexes, resulting in whats known as a multi-index

In [ ]:
countries_reviewed = reviews.groupby(['country', 'province']).description.agg([len])
countries_reviewed

In [ ]:
# Can also convert it simply back into a regular index using reset_index()
print(type(countries_reviewed.reset_index()))


## Sorting

Can get data in the order we want through sorting rather than relying on the default index sorting.

In [ ]:
countries_reviewed = countries_reviewed.reset_index()
countries_reviewed.sort_values(by='len')

In [ ]:
# Can also set to descending order
countries_reviewed.sort_values(by='len', ascending=False)

In [ ]:
# Can go back to default index sorting too
countries_reviewed.sort_index()

In [ ]:
# Can sort by more than one column at a time
countries_reviewed.sort_values(by=['country', 'len'])

## Data Types and Missing Values

### Data types
Use `.dtype` for a Series and `.dtypes` for a DataFrame. Types include `int64`, `float64`, string types, and `object`. Text may use an object or dedicated string dtype depending on the pandas version and how the data was constructed.

In [ ]:
# dtype for a given column
reviews.price.dtype

In [ ]:
# dtypes for a given DataFrame (Dont forget the 's'!)
reviews.dtypes

It's possible to convert a column of one type to another using astype()

In [ ]:
# Converts int64 data type column into float64
reviews.points.astype('float64')

## Dealing with missing data

Missing data may use `NaN`, `pd.NA`, or `NaT`, depending on the dtype. Use `isna()`/`isnull()` or `notna()`/`notnull()` to detect it.

In [ ]:
# pd.isnull and pd.notnull() are used for detecting null and not null values
# pd.isnull returns a Boolean mask; indexing below selects matching rows
reviews[pd.isnull(reviews.country)]

Pandas has operations for replacing missing values, such as with fillna(). Can selectively pick which value to replace NaN with. Another strategy is known as backfilling which consists of filling each missing value with the first non-null value that appears sometime after the given record in the database.

In [ ]:
# Replacing NaN with "Unknown"

reviews.region_2.fillna("Unknown")

As well can also replace specific non-null values we would like to replace.

In [ ]:
# Consider if a reviewer has changed their Twitter Handle
# We can replace their old handle with the new one as follows
reviews.taster_twitter_handle.replace("@kerinokeefe", "@kerino")

## Renaming and Combining

Can use pandas functions to change the names of entries to something better. Can also combine data from multiple DataFrames and/or Series.

### Renaming

rename() allows you to change the index names and/or column names

In [ ]:
# Changing 'points' column in the dataset to 'score'
reviews.rename(columns={'points': 'score'})

In [ ]:
# Can also use rename for renaming rows instead of columns
# by using 'index' param instead of 'column'
reviews.rename(index={0: 'firstEntry', 1: 'secondEntry'})

Note the column and row index have their own name attribute, which can be adjusted using rename_axis()

In [ ]:
reviews.rename_axis("wines", axis='rows').rename_axis("fields", axis='columns')

### Combining

`concat()` stacks objects along an axis; `join()` commonly combines on indexes; `merge()` explicitly matches key columns or indexes. Choose the operation and join type that match the intended relationship.

concat() is the simplest. Given a list of elements the function will smush the elements together along an axis.

This is useful when we have data in different DataFrames or Series but have the same fields (columns).

In [ ]:
# YT videos dataset splits the data up based on country of origin.
# We can study both Canada and UK by smushing them together using concat()

canadian_youtube = load_csv("CAvideos.csv")
british_youtube = load_csv("GBvideos.csv")

pd.concat([canadian_youtube, british_youtube])

`join()` combines DataFrames using their indexes by default. An inner join keeps only keys present in both inputs; the default left join also retains unmatched left-hand records. Repeated keys can produce multiple matched rows.

In [ ]:
# Set the index to a shared common key used for both datasets
left = canadian_youtube.set_index(['title', 'trending_date'])
right = british_youtube.set_index(['title', 'trending_date'])

# pulls videos that happened to be trending on the same day in both
# Canada AND the UK.
left.join(right, how='inner', lsuffix='_CAN', rsuffix='_UK')

## Resources
- https://www.kaggle.com/learn/pandas
- https://www.geeksforgeeks.org/pandas/pandas-tutorial/
- https://www.kaggle.com/datasets/zynicide/wine-reviews/data
- https://www.kaggle.com/datasets/datasnaek/youtube-new
- https://www.kaggle.com/datasets/residentmario/things-on-reddit/data